# Kaggle — 5.000 keyframe, 3 embedding models

**Models** (each processed **sequentially**, not simultaneously):

1. SigLIP 2 SO400M — `google/siglip2-so400m-patch14-384` (1152D)
2. SigLIP 2 Giant / SigLIP-G — `google/siglip2-giant-opt-patch16-384` (1536D)
3. Qwen3-VL-Embedding-2B — `Qwen/Qwen3-VL-Embedding-2B` (2048D)

**Input**: Kaggle dataset `batchkkk`, with `Batch1/Batch1/*.jpg`, `Batch2/...`, and `Batch3/...`. The scanner supports nested folders, and expects **ordinary JPG files**, not ZIPs.

**Output**: `/kaggle/working/BATCH_3_large_models_embeddings.zip`, with normalized float32 embeddings and a common `metadata.csv` shared across models. Partial checkpoints are written to `/kaggle/working/keyframe_embeddings_large/full/`.

**Important**: On **8GB VRAM**, the Giant checkpoint and Qwen may still run out of memory, especially under a memory-fragmented Kaggle environment. Start with `MAX_IMAGES = 12` (test), then set `None`. For Qwen, the default is a **4-bit quantized** model (not identical numerically to FP16). Do not mix vectors from different models.

**Internet ON**, GPU ON; **do not run this notebook with old checkpoints in the same kernel**.

In [ ]:
# CELL 1 — Dependencies. Run once (requires Kaggle Internet = ON)
%pip install -q --upgrade "transformers>=4.57.0,<5" "accelerate>=1.3" "qwen-vl-utils>=0.0.14" "bitsandbytes>=0.45" "safetensors" "pillow" "pandas" "tqdm"

In [ ]:
# CELL 2 — Configuration
import os, re, sys, gc, json, hashlib, subprocess, traceback, warnings, zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import transformers

INPUT_ROOT = Path('/kaggle/input/batchkkk')
if not INPUT_ROOT.exists():
    INPUT_ROOT = Path('/kaggle/input')  # handles alternative dataset mount

MAX_IMAGES = 12       # SMOKE TEST: 12. Change to None for ALL ~5,000 images.
RUN_KEYS = ['siglip2_so400m', 'siglip2_giant', 'qwen3_vl_2b']
QWEN_4BIT = True      # Recommended on 8GB VRAM. Set False to try full FP16.
QWEN_MAX_PIXELS = 384 * 384  # Reduce to 256*256 if Qwen OOMs
QWEN_MAX_LENGTH = 1024

MODELS = [
    {'key': 'siglip2_so400m', 'hf': 'google/siglip2-so400m-patch14-384', 'dim': 1152, 'batch_size': 2},
    {'key': 'siglip2_giant',  'hf': 'google/siglip2-giant-opt-patch16-384', 'dim': 1536, 'batch_size': 1},
    {'key': 'qwen3_vl_2b',   'hf': 'Qwen/Qwen3-VL-Embedding-2B', 'dim': 2048, 'batch_size': 1},
]

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Kaggle > Settings > Accelerator > GPU')
DEVICE = torch.device('cuda:0')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
RUN_SUFFIX = 'full' if MAX_IMAGES is None else f'test_{MAX_IMAGES}'
OUTPUT_ROOT = Path('/kaggle/working/keyframe_embeddings_large') / RUN_SUFFIX
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path('/kaggle/working/BATCH_3_large_models_embeddings.zip' if MAX_IMAGES is None else f'/kaggle/working/BATCH_3_large_models_embeddings_{RUN_SUFFIX}.zip')

print('INPUT_ROOT:', INPUT_ROOT, 'exists=', INPUT_ROOT.exists())
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('GPU:', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), 'GiB')
print('torch:', torch.__version__, 'transformers:', transformers.__version__, 'dtype:', DTYPE)
print('Model selection:', RUN_KEYS, '| Qwen4bit:', QWEN_4BIT)

In [ ]:
# CELL 3 — Scan JPG files and create stable shared metadata (NOT ZIP)
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
def natural_key(s):
    return [int(x) if x.isdigit() else x.casefold() for x in re.split(r'(\d+)', str(s))]

records = []
for img_path in INPUT_ROOT.rglob('*'):
    if not img_path.is_file() or img_path.suffix.lower() not in EXTENSIONS:
        continue
    batch = None
    for part in img_path.relative_to(INPUT_ROOT).parts[:-1]:
        mt = re.fullmatch(r'batch[ _-]*([123])', part, flags=re.IGNORECASE)
        if mt:
            batch = f'batch{mt.group(1)}'
            break
    if batch is None:
        continue
    rel = img_path.relative_to(INPUT_ROOT).as_posix()
    video_match = re.search(r'([a-z]\d{2}_v\d{3})', rel, flags=re.IGNORECASE)
    records.append({
        'batch': batch,
        'relative_path': rel,
        'image_path': str(img_path),
        'filename': img_path.name,
        'video_id': video_match.group(1) if video_match else '',
        'file_size': img_path.stat().st_size,
    })
records.sort(key=lambda r: (int(r['batch'][-1]), natural_key(r['relative_path'])))
if not records:
    raise FileNotFoundError(f'No image files found under {INPUT_ROOT}; check Add Input dataset')
if MAX_IMAGES is not None:
    records = records[:MAX_IMAGES]
metadata = pd.DataFrame(records)
metadata.insert(0, 'row_id', np.arange(len(metadata), dtype=int))
DATA_SIGNATURE = hashlib.sha256(metadata[['row_id','batch','relative_path','file_size']].to_csv(index=False).encode()).hexdigest()
metadata.to_csv(OUTPUT_ROOT / 'metadata.csv', index=False, encoding='utf-8-sig')
print('Images per batch:', metadata.groupby('batch').size().to_dict())
print('TOTAL:', len(metadata), '| signature:', DATA_SIGNATURE[:16])
if MAX_IMAGES is None and len(metadata) != 5000:
    warnings.warn(f'Found {len(metadata)} images (expected ~5000). Verify dataset structure.')
display(metadata.head())

In [ ]:
# CELL 4 — Model loaders (SigLIP uses get_image_features; Qwen uses official embedding-specific code)
from transformers import AutoModel, AutoProcessor, BitsAndBytesConfig

REPO_ROOT = Path('/kaggle/working/Qwen3-VL-Embedding')

def ensure_qwen_repo():
    if not (REPO_ROOT / 'src' / 'models' / 'qwen3_vl_embedding.py').exists():
        print('Cloning Qwen official embedding code...')
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/QwenLM/Qwen3-VL-Embedding.git', str(REPO_ROOT)],
            check=True,
        )
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))


def load_encoder(spec):
    hf = spec['hf']
    print('Loading', hf)
    if spec['key'] != 'qwen3_vl_2b':
        processor = AutoProcessor.from_pretrained(hf, use_fast=False)
        model = AutoModel.from_pretrained(
            hf, torch_dtype=DTYPE, low_cpu_mem_usage=True,
            attn_implementation='sdpa', device_map={'': 0},
        ).eval()
        return ('siglip', model, processor)

    # Qwen3-VL-Embedding is NOT Qwen3-VL-ForConditionalGeneration and should
    # NOT use the ordinary get_image_features() method from SigLIP.
    ensure_qwen_repo()
    from src.models.qwen3_vl_embedding import Qwen3VLEmbedder, Qwen3VLForEmbedding
    from transformers.models.qwen3_vl.processing_qwen3_vl import Qwen3VLProcessor

    if QWEN_4BIT:
        # Official Qwen3VLEmbedder.__init__ moves the model with .to(cuda), which
        # is incompatible with some bitsandbytes 4-bit configurations. Initialize
        # the exact same official embedder methods without that redundant .to().
        cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_use_double_quant=True,
        )
        helper = Qwen3VLEmbedder.__new__(Qwen3VLEmbedder)
        helper.max_length = QWEN_MAX_LENGTH
        helper.min_pixels = 4 * 32 * 32
        helper.max_pixels = QWEN_MAX_PIXELS
        helper.total_pixels = 7864320
        helper.fps = 1.0
        helper.max_frames = 64
        helper.default_instruction = "Represent the user's input."
        helper.model = Qwen3VLForEmbedding.from_pretrained(
            hf, quantization_config=cfg, torch_dtype=DTYPE,
            attn_implementation='sdpa', low_cpu_mem_usage=True,
            device_map={'': 0}, trust_remote_code=True,
        ).eval()
        helper.processor = Qwen3VLProcessor.from_pretrained(hf, padding_side='right')
    else:
        helper = Qwen3VLEmbedder(
            model_name_or_path=hf,
            max_length=QWEN_MAX_LENGTH,
            max_pixels=QWEN_MAX_PIXELS,
            torch_dtype=DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation='sdpa',
        )
    return ('qwen', helper, None)


@torch.inference_mode()
def encode_pil_images(encoder, pil_images):
    kind, model, processor = encoder
    if kind == 'siglip':
        inputs = processor(images=pil_images, return_tensors='pt')
        inputs = {k: v.to(DEVICE, dtype=DTYPE) if torch.is_tensor(v) and v.is_floating_point()
                  else v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}
        embs = model.get_image_features(**inputs)
    else:
        # Qwen official helper silently falls back to a NULL prompt if its vision
        # preprocessor fails. Check that an actual image can be processed so
        # we never accidentally save identical NULL-image vectors.
        from qwen_vl_utils.vision_process import process_vision_info
        conv = model.format_model_input(image=pil_images[0])
        checked_images, _, _ = process_vision_info(
            [conv], image_patch_size=16,
            return_video_metadata=True, return_video_kwargs=True,
        )
        if checked_images is None or len(checked_images) == 0:
            raise RuntimeError('Qwen vision preprocessing returned no image')
        # Official instruction-aware image embedding path: pooled last-token hidden state
        embs = model.process([{'image': im} for im in pil_images], normalize=True)
    if not isinstance(embs, torch.Tensor):
        raise TypeError(f'Unexpected embedding type: {type(embs)}')
    vectors = F.normalize(embs.float(), p=2, dim=-1).cpu().numpy().astype(np.float32)
    if vectors.ndim != 2 or vectors.shape[0] != len(pil_images) or not np.isfinite(vectors).all():
        raise ValueError('Invalid embedding output: shape/NaN/Inf')
    return vectors


def forward_adaptive(encoder, images):
    try:
        return encode_pil_images(encoder, images)
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        if len(images) == 1:
            raise RuntimeError('CUDA OOM at batch_size=1; try lower resolution/Qwen pixels or more VRAM.')
        mid = len(images)//2
        return np.concatenate([forward_adaptive(encoder, images[:mid]),
                               forward_adaptive(encoder, images[mid:])], axis=0)

In [ ]:
# CELL 5 — Resume-safe extraction, one model at a time

def atomic_json(path, data):
    temp = path.with_suffix('.tmp')
    temp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    temp.replace(path)


def extract_one(spec):
    key, n, dim = spec['key'], len(metadata), spec['dim']
    folder = OUTPUT_ROOT / key
    folder.mkdir(parents=True, exist_ok=True)
    em_file, valid_file = folder/'embeddings.npy', folder/'valid_mask.npy'
    state_file, errors_file = folder/'progress.json', folder/'errors.jsonl'
    identity = {
        'checkpoint': spec['hf'], 'dataset_signature': DATA_SIGNATURE,
        'rows': n, 'dim': dim,
        'quantized_4bit': bool(QWEN_4BIT) if key == 'qwen3_vl_2b' else False,
        'qwen_max_pixels': QWEN_MAX_PIXELS if key == 'qwen3_vl_2b' else None,
    }

    if state_file.exists():
        state = json.loads(state_file.read_text(encoding='utf-8'))
        if any(state.get(k) != v for k, v in identity.items()):
            raise RuntimeError(f'{key}: output incompatible with current settings. Use a new OUTPUT_ROOT or delete this model output folder intentionally.')
        if not em_file.exists() or not valid_file.exists():
            raise RuntimeError(f'{key}: incomplete checkpoint (missing .npy)')
        start = int(state['next_row'])
        emb = np.load(em_file, mmap_mode='r+')
        valid = np.load(valid_file, mmap_mode='r+')
        assert emb.shape == (n, dim) and valid.shape == (n,)
        if start >= n:
            print(f'{key}: already DONE — skip.')
            return {'model': key, 'status': 'complete', 'valid': int(valid.sum()), 'total': n}
    else:
        if em_file.exists() or valid_file.exists():
            raise RuntimeError(f'{key}: old .npy files without progress.json; inspect before deleting.')
        start = 0
        emb = np.lib.format.open_memmap(em_file, mode='w+', dtype='float32', shape=(n, dim))
        valid = np.lib.format.open_memmap(valid_file, mode='w+', dtype='bool', shape=(n,))
        emb[:] = 0
        valid[:] = False
        emb.flush(); valid.flush()
        atomic_json(state_file, {**identity, 'next_row': 0})

    encoder = None
    try:
        encoder = load_encoder(spec)
        print(f'GPU allocated after load: {torch.cuda.memory_allocated()/2**30:.2f} GiB')
        for left in tqdm(range(start, n, spec['batch_size']), desc=key):
            right = min(left + spec['batch_size'], n)
            pictures, positions, errors = [], [], []
            for idx in range(left, right):
                path = metadata.iloc[idx]['image_path']
                try:
                    with Image.open(path) as im:
                        pic = ImageOps.exif_transpose(im).convert('RGB')
                    pictures.append(pic)
                    positions.append(idx)
                except Exception as ex:
                    errors.append({'row_id': int(idx), 'image': path, 'error': repr(ex)})
            try:
                if pictures:
                    vectors = forward_adaptive(encoder, pictures)
                    if vectors.shape != (len(pictures), dim):
                        raise ValueError(f'{key}: unexpected dimension {vectors.shape}, expected {dim}')
                    emb[positions] = vectors
                    valid[positions] = True
            finally:
                for pic in pictures:
                    pic.close()
            if errors:
                with errors_file.open('a', encoding='utf-8') as log:
                    for error in errors:
                        log.write(json.dumps(error, ensure_ascii=False) + '\n')
            emb.flush(); valid.flush()
            atomic_json(state_file, {**identity, 'next_row': right})

        valid_count = int(np.count_nonzero(valid))
        print(f'DONE {key}: {valid_count}/{n} valid, embedding shape={emb.shape}')
        return {'model': key, 'status': 'complete', 'valid': valid_count, 'total': n}
    finally:
        if emb is not None: emb.flush()
        if valid is not None: valid.flush()
        del emb, valid, encoder
        gc.collect()
        torch.cuda.empty_cache()


RESULTS = []
for spec in MODELS:
    if spec['key'] not in RUN_KEYS:
        continue
    try:
        RESULTS.append(extract_one(spec))
    except Exception as ex:
        print(f'FAILED [{spec["key"]}]: {type(ex).__name__}: {ex}')
        traceback.print_exc(limit=6)
        RESULTS.append({'model': spec['key'], 'status': 'failed', 'error': str(ex)[:1000], 'total': len(metadata)})
        gc.collect()
        torch.cuda.empty_cache()

pd.DataFrame(RESULTS).to_csv(OUTPUT_ROOT/'run_results.csv', index=False, encoding='utf-8-sig')
display(pd.DataFrame(RESULTS))

In [ ]:
# CELL 6 — Validate outputs, make per-batch arrays, and package ZIP
summary = []
for spec in MODELS:
    folder = OUTPUT_ROOT/spec['key']
    progress_path = folder/'progress.json'
    if not progress_path.exists():
        summary.append({'model': spec['key'], 'status': 'not_started'})
        continue
    state = json.loads(progress_path.read_text(encoding='utf-8'))
    if state.get('dataset_signature') != DATA_SIGNATURE or state.get('checkpoint') != spec['hf']:
        summary.append({'model': spec['key'], 'status': 'signature_mismatch'})
        continue
    complete = state.get('next_row') == len(metadata)
    if not complete:
        summary.append({'model': spec['key'], 'status': f'incomplete: {state.get("next_row")}/{len(metadata)}'})
        continue
    emb = np.load(folder/'embeddings.npy', mmap_mode='r')
    mask = np.load(folder/'valid_mask.npy', mmap_mode='r')
    if emb.shape != (len(metadata), spec['dim']) or mask.shape != (len(metadata),):
        raise RuntimeError(f'{spec["key"]}: bad output shape')
    valid_ids = np.flatnonzero(mask)
    if len(valid_ids):
        sample = np.asarray(emb[valid_ids[:min(20,len(valid_ids))]])
        norms = np.linalg.norm(sample, axis=1)
        if not np.isfinite(sample).all() or not np.allclose(norms, 1.0, atol=1e-3):
            raise RuntimeError(f'{spec["key"]}: invalid normalized embeddings')
    for batch in ('batch1', 'batch2', 'batch3'):
        ids = metadata.index[metadata['batch'].eq(batch)].to_numpy()
        np.save(folder/f'{batch}.npy', np.asarray(emb[ids], dtype=np.float32))
        np.save(folder/f'{batch}_valid_mask.npy', np.asarray(mask[ids], dtype=bool))
    summary.append({'model': spec['key'], 'status': 'complete', 'valid': int(mask.sum()), 'total': len(mask), 'dim': emb.shape[1]})
    del emb, mask
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUTPUT_ROOT/'summary.csv', index=False, encoding='utf-8-sig')
manifest = {
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'input_root': str(INPUT_ROOT),
    'data_signature': DATA_SIGNATURE,
    'total_images': int(len(metadata)),
    'model_results': summary,
    'notes': 'Embeddings are L2-normalized float32. row_id in metadata.csv is the common array index. qwen3_vl_2b is NF4 4-bit when enabled.',
}
(OUTPUT_ROOT/'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(summary_df.to_string(index=False))

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for file in OUTPUT_ROOT.rglob('*'):
        if file.is_file() and file.suffix != '.tmp':
            archive.write(file, arcname=file.relative_to(OUTPUT_ROOT).as_posix())
print('ZIP:', ZIP_PATH, '| size MiB:', round(ZIP_PATH.stat().st_size / 2**20, 1))
print('If a model is incomplete, rerun CELL 5, then rerun CELL 6. Do not interpret partial arrays as complete results.')

## Loading embeddings after extraction

```python
import numpy as np, pandas as pd
from pathlib import Path
root = Path('/kaggle/working/keyframe_embeddings_large/full')
meta = pd.read_csv(root / 'metadata.csv')
siglip400 = np.load(root / 'siglip2_so400m' / 'embeddings.npy', mmap_mode='r')
print(siglip400.shape, meta.loc[0, 'image_path'])
```

**Important**: all three embedding spaces are independent; a text query must be encoded with **the same checkpoint** and the **same Qwen quantization/prompt policy** as the image index. `row_id` is the zero-based row index in every `embeddings.npy` file. Missing/unreadable images have `valid_mask=False`.

**Qwen resolution trade-off**: `QWEN_MAX_PIXELS=384*384` attempts to preserve details; `256*256` lowers memory needs but may reduce color/attribute retrieval performance. Qwen-Giant and 2B are not guaranteed to fit every 8GB GPU runtime; failures are captured in `run_results.csv` and other models still run.